In [1]:
import numpy as np
import pandas as pd

In [29]:
dtype_map = {
    "Task Id": "string",
    "FPOGV": "Int64", "FPOGID": "Int64",
    "LPMMV": "Int64", "RPMMV": "Int64",
    "BKID": "Int64",  "BKDUR": "float64", 
    "AOI": "string"
}
fixations_onoff = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fix_onoff_task1.csv", dtype=dtype_map)
gaze_onoff = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/gaze_onoff_task1.csv", dtype=dtype_map)

In [37]:
# Gazepoint TIMETICK frequency
TICK_HZ = 10_000_000  # TIMETICK(f=10000000)

# -------------------------------------------------------
# A) Windowing & label refinement
# -------------------------------------------------------
def build_shared_anchor(
    *dfs,
    tick_col: str = "TIMETICK(f=10000000)",
    group_cols=("Video Name (folder)", "User ID (file)")
) -> pd.DataFrame:
    """
    Compute one shared 'anchor_tick' per (Video, User) across any number of tables.
    The anchor is the minimum TIMETICK seen across those tables for that (Video, User).
    """
    refs = []
    for d in dfs:
        cols = list(group_cols) + [tick_col]
        tmp = d[cols].copy()
        tmp[tick_col] = pd.to_numeric(tmp[tick_col], errors="coerce")
        refs.append(
            tmp.groupby(list(group_cols), as_index=False)[tick_col]
               .min()
               .rename(columns={tick_col: "min_tick"})
        )
    ref = (
        pd.concat(refs, ignore_index=True)
          .groupby(list(group_cols), as_index=False)["min_tick"].min()
          .rename(columns={"min_tick": "anchor_tick"})
    )
    return ref

def add_window_id_by_timetick(
    df: pd.DataFrame,
    *,
    win_sec: float,
    tick_col: str = "TIMETICK(f=10000000)",
    group_cols=("Video Name (folder)", "User ID (file)"),
    shared_ref: pd.DataFrame | None = None,
    out_col: str = "win_id",
) -> pd.DataFrame:
    """
    Assign an integer window id per (Video, User) by time from anchor, in fixed-length windows.
    """
    out = df.copy()
    out[tick_col] = pd.to_numeric(out[tick_col], errors="coerce")
    win_ticks = int(round(win_sec * TICK_HZ))

    if shared_ref is not None:
        out = out.merge(shared_ref, on=list(group_cols), how="left", validate="many_to_one")
        anchor = out["anchor_tick"]
    else:
        anchor = out.groupby(list(group_cols))[tick_col].transform("min")

    out[out_col] = ((out[tick_col] - anchor) // win_ticks).astype("Int64")
    return out

def refine_window_labels_by_fraction(
    df_with_win: pd.DataFrame,
    *,
    group_cols=("Video Name (folder)", "User ID (file)"),
    win_col="win_id",
    label_col="label",
    on_hi=0.80,
    off_lo=0.20,
    out_col="label",
) -> pd.DataFrame:
    """
    Denoise window labels by ON/OFF fraction within each (Video,User,win_id).
    Keep only windows confidently ON or OFF.
    """
    d = df_with_win.copy()
    d[label_col] = d[label_col].astype(str).str.upper().str.strip()
    d = d[d[label_col].isin(["ON", "OFF"])].copy()

    key = list(group_cols) + [win_col]
    d["_is_on"] = (d[label_col] == "ON").astype(float)

    frac = (
        d.groupby(key, dropna=False)["_is_on"].mean()
         .rename("on_fraction").reset_index()
    )

    frac["label_refined"] = np.where(
        frac["on_fraction"] >= on_hi, "ON",
        np.where(frac["on_fraction"] <= off_lo, "OFF", np.nan)
    )

    d = d.merge(frac[key + ["label_refined", "on_fraction"]],
                on=key, how="left", validate="many_to_one")

    d = d[~d["label_refined"].isna()].copy()
    d[out_col] = d["label_refined"]
    d.drop(columns=["_is_on", "label_refined"], inplace=True)
    return d

# -------------------------------------------------------
# B) Feature helpers
# -------------------------------------------------------
def _mad(x):
    x = pd.Series(x, dtype="float64").dropna()
    if x.empty: return np.nan
    med = x.median()
    return (x - med).abs().median()

def _pupil_slope(tick, pupil):
    # robust slope via np.polyfit on valid points; time in seconds from first sample
    if tick is None:
        return np.nan
    tt = pd.Series(tick, dtype="float64")
    pp = pd.Series(pupil, dtype="float64")
    m = (~tt.isna()) & (~pp.isna())
    if m.sum() < 5:
        return np.nan
    tsec = (tt[m] - tt[m].iloc[0]) / TICK_HZ
    try:
        slope, _ = np.polyfit(tsec.values, pp[m].values, 1)
    except Exception:
        slope = np.nan
    return slope

def _circular_variance_deg(angles_deg):
    a = pd.Series(angles_deg, dtype="float64").dropna()
    if a.empty: return np.nan
    rad = np.deg2rad(a.values)
    C = np.mean(np.cos(rad))
    S = np.mean(np.sin(rad))
    R = np.sqrt(C*C + S*S)
    return 1 - R  # 0=tightly clustered, 1=uniform

def _valid_mask(w: pd.DataFrame) -> pd.Series:
    return (w.get("FPOGV", 1) == 1)

def _safe_std(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(s.std()) if len(s) else np.nan

def _down_ratio_from_dirs(dir_deg: pd.Series) -> float:
    """
    Heuristic: y increases downward; 'down' ≈ [180°, 360°) in standard screen polar.
    """
    d = pd.to_numeric(dir_deg, errors="coerce").dropna()
    if d.empty:
        return np.nan
    return float(((d % 360) >= 180).mean())

def _count_content_to_footer(aoi_series: pd.Series, aoi_role_map: dict | None) -> int:
    """Count AOI transitions from role='CONTENT' to role='FOOTER'."""
    if aoi_series is None:
        return 0
    a = aoi_series.astype(str).str.strip()
    if a.empty:
        return 0
    lower = a.str.lower()
    a = a[~lower.isin(["", "none", "null", "nan"])]
    if a.empty:
        return 0

    if aoi_role_map is None:
        def to_role(x: str) -> str:
            xl = x.lower()
            if "foot" in xl: return "FOOTER"
            if "head" in xl or "nav" in xl or "menu" in xl: return "HEADER"
            return "CONTENT"
        roles = a.map(to_role)
    else:
        roles = a.map(lambda x: aoi_role_map.get(x, "CONTENT"))

    roles_compact = roles[roles.shift(1) != roles]
    return int(((roles_compact.shift(1) == "CONTENT") & (roles_compact == "FOOTER")).sum())

# -------------------------------------------------------
# C) Windowed features
# -------------------------------------------------------
def window_features_fixations(fix_df: pd.DataFrame, *, win_sec: float) -> pd.DataFrame:
    """
    Fixation features per window:
      - fix_efficiency, fix_aoi_ratio, fix_rate_per_s, fix_median_dur
    """
    keys = ["Video Name (folder)", "User ID (file)", "Task Id", "type", "label", "phase", "block_id", "win_id"]
    eps = 1e-6

    def _agg(w: pd.DataFrame) -> pd.Series:
        fix_count      = w["FPOGID"].nunique() if "FPOGID" in w else np.nan
        fix_mean_dur   = w["FPOGD"].mean()     if "FPOGD"  in w else np.nan
        fix_median_dur = w["FPOGD"].median()   if "FPOGD"  in w else np.nan
        fix_aoi_ratio  = w["AOI"].notna().mean() if "AOI" in w else np.nan

        fix_efficiency = (fix_count / (abs(fix_mean_dur) + eps)) if pd.notna(fix_mean_dur) else np.nan
        fix_rate_per_s = (fix_count / win_sec) if pd.notna(fix_count) else np.nan

        return pd.Series({
            "fix_efficiency":  fix_efficiency,
            "fix_aoi_ratio":   fix_aoi_ratio,
            "fix_rate_per_s":  fix_rate_per_s,
            "fix_median_dur":  fix_median_dur,
        })

    return (
        fix_df.dropna(subset=["win_id"])
              .groupby(keys, dropna=False)
              .apply(_agg)
              .reset_index()
    )

def window_features_gaze(gaze_df: pd.DataFrame, *, win_sec: float = 3.0) -> pd.DataFrame:
    """
    Gaze/pupil/saccade/blink features per window.
    """
    keys = ["Video Name (folder)", "User ID (file)", "Task Id", "type", "label", "phase", "block_id", "win_id"]

    def _agg(w: pd.DataFrame) -> pd.Series:
        gv = (w["FPOGV"] == 1) if "FPOGV" in w else pd.Series(True, index=w.index)
        lv = (w["LPMMV"] == 1) if "LPMMV" in w else pd.Series(False, index=w.index)
        rv = (w["RPMMV"] == 1) if "RPMMV" in w else pd.Series(False, index=w.index)

        # Dispersion
        gx = w.loc[gv, "FPOGX"].std() if "FPOGX" in w else np.nan
        gy = w.loc[gv, "FPOGY"].std() if "FPOGY" in w else np.nan
        gaze_disp_total = (
            np.sqrt((0 if pd.isna(gx) else gx**2) + (0 if pd.isna(gy) else gy**2))
            if (not pd.isna(gx) or not pd.isna(gy)) else np.nan
        )

        # Pupil
        if "LPMM" in w and "RPMM" in w:
            l = w["LPMM"].where(lv)
            r = w["RPMM"].where(rv)
            pupil_row = pd.concat([l, r], axis=1).mean(axis=1, skipna=True)
            pupil_mean  = pupil_row.mean()
            pupil_var   = pupil_row.var(ddof=1)
            pupil_mad   = _mad(pupil_row)
            tick_col = "TIMETICK(f=10000000)"
            pupil_slope = _pupil_slope(w[tick_col] if tick_col in w else None, pupil_row)
        else:
            pupil_mean = pupil_var = pupil_mad = pupil_slope = np.nan

        # Saccades
        saccade_mean_mag = w.loc[gv, "SACCADE_MAG"].mean() if "SACCADE_MAG" in w else np.nan
        if "SACCADE_DIR" in w:
            saccade_circ_var = _circular_variance_deg(w.loc[gv, "SACCADE_DIR"])
            saccade_dir_var  = w.loc[gv, "SACCADE_DIR"].var(ddof=1)
        else:
            saccade_circ_var = np.nan
            saccade_dir_var  = np.nan
        if pd.isna(saccade_dir_var):
            saccade_dir_var = 0.0

        # Blinks & quality
        blink_count      = w.loc[gv, "BKID"].nunique() if "BKID" in w else np.nan
        blink_rate       = (blink_count / win_sec) if pd.notna(blink_count) else np.nan
        valid_gaze_ratio = gv.mean() if len(w) else np.nan

        return pd.Series({
            "gaze_dispersion_total": gaze_disp_total,
            "pupil_mean":            pupil_mean,
            "pupil_var":             0 if pd.isna(pupil_var) else pupil_var,
            "pupil_mad":             pupil_mad,
            "pupil_slope":           pupil_slope,
            "saccade_mean_mag":      saccade_mean_mag,
            "saccade_dir_var":       0 if pd.isna(saccade_dir_var) else saccade_dir_var,
            "saccade_circ_var":      saccade_circ_var,
            "blink_rate":            blink_rate,
            "valid_gaze_ratio":      valid_gaze_ratio,
        })

    return (
        gaze_df.dropna(subset=["win_id"])
               .groupby(keys, dropna=False)
               .apply(_agg)
               .reset_index()
    )

def window_features_scroll(
    gaze_df: pd.DataFrame, 
    *, 
    win_sec: float, 
    aoi_role_map: dict | None = None
) -> pd.DataFrame:
    """
    Scroll-specific features per window:
      - vert_frac
      - net_vert_disp (y_end - y_start)  [0=top,1=bottom → positive ≈ downward]
      - prop_down_saccades
      - fix_y_over_x (std_y / std_x)
      - trans_content_to_footer  (needs AOI; heuristic mapping if none given)
    """
    keys = ["Video Name (folder)", "User ID (file)", "Task Id", "type", "label", "phase", "block_id", "win_id"]
    df = gaze_df.dropna(subset=["win_id"]).copy()

    def _agg(w: pd.DataFrame) -> pd.Series:
        vmask = _valid_mask(w)
        x = pd.to_numeric(w.loc[vmask, "FPOGX"], errors="coerce") if "FPOGX" in w else pd.Series(dtype=float)
        y = pd.to_numeric(w.loc[vmask, "FPOGY"], errors="coerce") if "FPOGY" in w else pd.Series(dtype=float)
        x = x.dropna(); y = y.dropna()

        # 1) vertical fraction & 2) net vertical displacement
        if len(x) >= 2 and len(y) >= 2:
            dx = np.diff(x.values)
            dy = np.diff(y.values)
            denom = np.sum(np.abs(dx)) + np.sum(np.abs(dy))
            vert_frac = float(np.sum(np.abs(dy)) / denom) if denom > 0 else np.nan
            net_vert_disp = float(y.iloc[-1] - y.iloc[0])
        else:
            vert_frac = np.nan
            net_vert_disp = np.nan

        # 3) proportion of downward saccades
        if "SACCADE_DIR" in w and w["SACCADE_DIR"].notna().any():
            prop_down_saccades = _down_ratio_from_dirs(w.loc[vmask, "SACCADE_DIR"])
        else:
            if len(y) >= 2:
                dy = np.diff(y.values)
                prop_down_saccades = float((dy > 0).mean()) if dy.size else np.nan
            else:
                prop_down_saccades = np.nan

        # 4) fixation geometry ratio
        sx = _safe_std(w.loc[vmask, "FPOGX"]) if "FPOGX" in w else np.nan
        sy = _safe_std(w.loc[vmask, "FPOGY"]) if "FPOGY" in w else np.nan
        eps = 1e-9
        fix_y_over_x = float(sy / (sx + eps)) if (pd.notna(sx) and pd.notna(sy)) else np.nan

        # 5) AOI transition Content -> Footer
        if "AOI" in w:
            trans_content_to_footer = _count_content_to_footer(w["AOI"], aoi_role_map)
        else:
            trans_content_to_footer = 0

        return pd.Series({
            "vert_frac": vert_frac,
            "net_vert_disp": net_vert_disp,
            "prop_down_saccades": prop_down_saccades,
            "fix_y_over_x": fix_y_over_x,
            "trans_content_to_footer": trans_content_to_footer,
        })

    return (
        df.groupby(keys, dropna=False)
          .apply(_agg)
          .reset_index()
    )

# -------------------------------------------------------
# D) Merge all features
# -------------------------------------------------------
def combine_fix_gaze_scroll(
    fix_df: pd.DataFrame,
    gaze_df: pd.DataFrame,
    *,
    win_sec: float = 3.0,
    aoi_role_map: dict | None = None
) -> pd.DataFrame:
    """
    Convenience: compute & merge fixation, gaze, and scroll features using the same windows.
    Expects input dataframes already contain 'win_id' and task/meta columns.
    """
    fix_w   = window_features_fixations(fix_df, win_sec=win_sec)
    gaze_w  = window_features_gaze(gaze_df,    win_sec=win_sec)
    scroll_w= window_features_scroll(gaze_df,  win_sec=win_sec, aoi_role_map=aoi_role_map)

    keys = ["Video Name (folder)", "User ID (file)", "Task Id", "type", "label", "phase", "block_id", "win_id"]
    m = pd.merge(fix_w,  gaze_w,  on=keys, how="outer", validate="one_to_one")
    m = pd.merge(m,      scroll_w, on=keys, how="outer", validate="one_to_one")
    return m

# =========================
# Example usage (uncomment & adapt)
# =========================
# Suppose you have raw samples for Task 1 in a DataFrame `raw` with columns:
# MEDIA_NAME, TIMETICK(f=10000000), FPOGX, FPOGY, FPOGV, FPOGD, FPOGID,
# LPMM, LPMMV, RPMM, RPMMV, AOI, BKID, BKDUR, SACCADE_MAG, SACCADE_DIR,
# and meta columns: "Video Name (folder)", "User ID (file)", "Task Id", "type", "label", "phase", "block_id"


/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_10483/1840883824.py:202: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_10483/1840883824.py:270: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_10483/1840883824.py:340: DeprecationWarning: DataFrameGroupBy.app

In [ ]:
raw = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fix_onoff_task1.csv")
win_len = 3.0  # seconds
ref = build_shared_anchor(raw, tick_col="TIMETICK(f=10000000)")
raw = add_window_id_by_timetick(raw, win_sec=win_len, shared_ref=ref)
raw_refined = refine_window_labels_by_fraction(raw, win_col="win_id", label_col="label", on_hi=0.80, off_lo=0.20)
#
# # If you keep fixations and gaze in the same `raw_refined`, you can pass the same df twice:
features_task1 = combine_fix_gaze_scroll(raw_refined, raw_refined, win_sec=win_len, aoi_role_map=None)
# display(features_task1.head())

In [38]:
features_task1.to_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/improving models/outputs1/fix_wid_t1.csv", index=False)    